<a href="https://colab.research.google.com/github/Anshuman22coder/BUG_FEEDBACK/blob/main/Training_deepseek_distilled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l9aaqvdn/unsloth_7d100e70c5914c96b74b6420a6d7f0f0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l9aaqvdn/unsloth_7d100e70c5914c96b74b6420a6d7f0f0
  Resolved https://github.com/unslothai/unsloth.git to commit 0003f889e6b8dd0a8da21cb382f175dab17985ac
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.9 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# === CHANGED HERE: Updated to match your exact testing system prompt ===
def format_for_qwen(example):
    return {
        "text": (
            "<|im_start|>system\n"
            "You are a Socratic tutor. Review the code carefully. "
            "First think about the logic flaws or if the code is correct, then provide feedback using Status, Analogy, Question. "
            "CRITICAL: The code might be 100% correct. If the code is already perfect, "
            "state that it is correct in the Status, and ask a conceptual follow-up question.\n<|im_end|>\n"
            f"<|im_start|>user\nCode to review:\n{example['code']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{example['thinking']}\n</think>\n"
            f"Status: {example['status']}\n"
            f"Analogy: {example['analogy']}\n"
            f"Question: {example['question']}\n<|im_end|>"
        )
    }

max_seq_length = 2048 # 2048 tokens (short term memory of the model.)
dtype = None
load_in_4bit = True

# 1. Load the pre-quantized base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 2. Apply LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# 3. Load and map your custom dataset
dataset = load_dataset("json", data_files="dataset.json", split="train")
dataset = dataset.map(format_for_qwen)

# 4. Configure the Supervised Fine-Tuning Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,# Instead of running backpropagation at every individual step, it aggregates the gradients over multiple steps.
        #You can change this to "cosine". A cosine curve decays slowly at first, drops quickly in the middle, and flattens out near the end. Cosine schedules often yield slightly better language generation performance but can be less forgiving if training terminates prematurely.
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.05,#What it means: A regularization penalty applied to large model weights to discourage overfitting
        #If your model starts memorizing your training set exactly but fails to generalize to real user questions, increase this to 0.05 or 0.1 to force the model to seek simpler, more generalized logic paths.


        lr_scheduler_type="linear",#After the warm-up phase peaks, the learning rate drops in a straight linear line down to absolute zero by the end of training.

        seed=3407,
        dataset_text_field="text",
        max_length=max_seq_length,#what i am giving the prompt, that's size..
        dataset_num_proc=2,
        packing=False,#What it means: Determines whether short data instances are grouped together into a single block of your maximum context length (max_length).

#Changing it: * Set to True: Stitches multiple short sequences together separated by EOS tokens. It makes training significantly faster because it minimizes padding tokens.

#Set to False (Your choice): Treats every sample separately. This is highly recommended for structured chat sequences or specialty data (like your Socratic reasoning dataset) so the model respects distinct sequence limits cleanly.
    ),
)

# 5. Run the training execution
print("🚀 Starting fine-tuning loop on Colab GPU...")
trainer_stats = trainer.train()

# 6. Save the newly adjusted LoRA adapter weights
model.save_pretrained("socratic_DeepSeek-R1-Distill-Qwen-7B-bnb-4bit_adapter")
tokenizer.save_pretrained("socratic_DeepSeek-R1-Distill-Qwen-7B-bnb-4bit_adapter")
print("✅ Training complete! Adapter saved to your local Colab directory.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,027 | Num Epochs = 1 | Total steps = 254
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.216992
2,3.137087
3,3.047498
4,3.118246
5,2.922304
6,2.675213
7,2.503446
8,2.432858
9,2.184279
10,2.045061


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-254/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in socratic_DeepSeek-R1-Distill-Qwen-7B-bnb-4bit_adapter/tokenizer_config.json.


✅ Training complete! Adapter saved to your local Colab directory.


In [ ]:
# 1. Switch the model to fast inference mode
FastLanguageModel.for_inference(model)

tricky_dp_code = """
import heapq

def find_shortest_path(n, edges, start, end):
    graph = [[] for _ in range(n)]
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))

    queue = [(0, start)]
    visited = [False] * n
    visited[start] = True

    while queue:
        dist, node = heapq.heappop(queue)
        if node == end:
            return dist
        for neighbor, weight in graph[node]:
            if not visited[neighbor]:
                visited[neighbor] = True
                heapq.heappush(queue, (dist + weight, neighbor))
    return -1
"""

# 2. Format using your identical merged training template
prompt_tokens = tokenizer(
    [
        "<|im_start|>system\n"
        "You are a Socratic tutor. Review the code carefully. "
        "First think about the logic flaws or if the code is correct, then provide feedback using Status, Analogy, Question. "
        "CRITICAL: The code might be 100% correct. If the code is already perfect, "
        "state that it is correct in the Status, and ask a conceptual follow-up question.\n<|im_end|>\n"
        f"<|im_start|>user\nCode to review:\n{tricky_dp_code}\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    ],
    return_tensors="pt",
).to("cuda")

# 3. Stream the execution
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("🧪 Testing Fine-Tuned Model on TRICKY DP Code:\n" + "="*40)
_ = model.generate(
    **prompt_tokens,
    streamer=text_streamer,
    max_new_tokens=512,
    stop_strings=["<|im_end|>"],
    tokenizer=tokenizer,
    use_cache=True,
    temperature=0.0,
    do_sample=False
)
print("\n" + "="*40)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Testing Fine-Tuned Model on TRICKY DP Code:
<think>
The user has provided a code snippet written in Python covering the topic of 'graphs'. Let's trace the logic line-by-line:
```
# 1006. Shortest Path
def find_shortest_path(n, edges, start, end):
    graph = [[] for _ in range(n)]
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))
    
    queue = [(0, start)]
    visited = [False] * n
    visited[start] = True
    
    while queue:
        dist, node = heapq.heappop(queue)
        if node == end:
            return dist
        for neighbor, weight in graph[node]:
            if not visited[neighbor]:
                visited[neighbor] = True
                heapq.heappush(queue, (dist + weight, neighbor))
    return -1
``` Analysis: Tracing the instruction sequence verifies that all loop invariants hold true, operations remain fully inside container bounds, and termination logic is cleanly handled. The implementation functions as intended. To a

In [ ]:
# Save the merged model directly as a quantized GGUF file
# Options for quantization_method: "q4_k_m" (fastest) or "q8_0" (retains higher intelligence)
model.save_pretrained_gguf(
    "socratic_3b_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/822 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in socratic_3b_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [04:35<13:45, 275.25s/it]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# === CHANGED HERE: Updated to match your exact testing system prompt ===
def format_for_qwen(example):
    return {
        "text": (
            "<|im_start|>system\n"
            "You are a Socratic tutor. Review the code carefully. "
            "First think about the logic flaws or if the code is correct, then provide feedback using Status, Analogy, Question. "
            "CRITICAL: The code might be 100% correct. If the code is already perfect, "
            "state that it is correct in the Status, and ask a conceptual follow-up question.\n<|im_end|>\n"
            f"<|im_start|>user\nCode to review:\n{example['code']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{example['thinking']}\n</think>\n"
            f"Status: {example['status']}\n"
            f"Analogy: {example['analogy']}\n"
            f"Question: {example['question']}\n<|im_end|>"
        )
    }

max_seq_length = 2048 # 2048 tokens (short term memory of the model.)
dtype = None
load_in_4bit = True

# 1. Load the pre-quantized base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 2. Apply LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# 3. Load and map your custom dataset
dataset = load_dataset("json", data_files="dataset.json", split="train")
dataset = dataset.map(format_for_qwen)

# 4. Configure the Supervised Fine-Tuning Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,# Instead of running backpropagation at every individual step, it aggregates the gradients over multiple steps.
        #You can change this to "cosine". A cosine curve decays slowly at first, drops quickly in the middle, and flattens out near the end. Cosine schedules often yield slightly better language generation performance but can be less forgiving if training terminates prematurely.
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,#What it means: A regularization penalty applied to large model weights to discourage overfitting
        #If your model starts memorizing your training set exactly but fails to generalize to real user questions, increase this to 0.05 or 0.1 to force the model to seek simpler, more generalized logic paths.


        lr_scheduler_type="linear",#After the warm-up phase peaks, the learning rate drops in a straight linear line down to absolute zero by the end of training.

        seed=3407,
        dataset_text_field="text",
        max_length=max_seq_length,#what i am giving the prompt, that's size..
        dataset_num_proc=2,
        packing=False,#What it means: Determines whether short data instances are grouped together into a single block of your maximum context length (max_length).

#Changing it: * Set to True: Stitches multiple short sequences together separated by EOS tokens. It makes training significantly faster because it minimizes padding tokens.

#Set to False (Your choice): Treats every sample separately. This is highly recommended for structured chat sequences or specialty data (like your Socratic reasoning dataset) so the model respects distinct sequence limits cleanly.
    ),
)

# 5. Run the training execution
print("🚀 Starting fine-tuning loop on Colab GPU...")
trainer_stats = trainer.train()

# 6. Save the newly adjusted LoRA adapter weights
model.save_pretrained("socratic_DeepSeek-R1-Distill-Qwen-7B-bnb-4bit_adapter")
tokenizer.save_pretrained("socratic_DeepSeek-R1-Distill-Qwen-7B-bnb-4bit_adapter")
print("✅ Training complete! Adapter saved to your local Colab directory.")